# 01 — Steel DSM Model
Loads raw data, runs the full model pipeline, saves all results to `data/processed/`.
**Run this file first**, then open `02_plots.ipynb`.

## Imports

In [105]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from dynamic_stock_model import DynamicStockModel

output_folder = Path.cwd().parent / 'data' / 'processed'
output_folder.mkdir(parents=True, exist_ok=True)
print('Output folder:', output_folder)

Output folder: c:\Users\ovid\MasterThesis\master\electrolysers\data\processed


## 1. Load raw data

In [106]:
base_path    = Path.cwd().parent
steel_folder = base_path / 'data' / 'raw' / 'Steel'
pop_folder   = base_path / 'data' / 'raw' / 'population'

steel_data      = pd.read_excel(steel_folder / 'steel.xlsx')
steel_shares    = pd.read_csv(steel_folder / 'steel_shares_1900_2008.csv')
pop_data        = pd.read_csv(pop_folder / 'population.csv')
future_pop_data = pd.read_csv(pop_folder / 'future_population.csv')
print('Raw data loaded.')

Raw data loaded.


C:\Users\ovid\AppData\Local\Temp\ipykernel_31096\1119624132.py:8: DtypeWarning: Columns (2,3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  future_pop_data = pd.read_csv(pop_folder / 'future_population.csv')


## 2. Clean steel stock data

In [107]:
steel_stock_data = steel_data.copy()
steel_stock_data.columns = steel_data.iloc[1]
steel_stock_data = steel_stock_data.drop(index=[0, 1])
steel_stock_data = steel_stock_data.rename(columns={
    'total stock S': 'steel_stock', 'S - Vehicles': 'vehicles',
    'S - Machinery': 'machinery',   'S - BC': 'BC', 'S - Appliances': 'appliances'
})
print('Steel data cleaned.')

Steel data cleaned.


## 3. Clean population data

In [108]:
# Historic population data
pop_data = pop_data[['Year', 'Code', 'Population']].copy()
pop_data = pop_data[pop_data['Code'] == 'OWID_WRL'].drop(columns=['Code'])
pop_data = pop_data.rename(columns={'Year': 'year', 'Population': 'population'})
pop_data = pop_data[(pop_data['year'] >= 1700) & (pop_data['year'] <= 2008)]

In [109]:
# Future population data
future_pop_data = future_pop_data[['Time', 'Location', 'TPopulation1Jan']].copy()
future_pop_data = future_pop_data[future_pop_data['Location'] == 'World']
future_pop_data = future_pop_data[(future_pop_data['Time'] >= 2009) & (future_pop_data['Time'] <= 2100)]
future_pop_data = future_pop_data.rename(columns={'Time': 'year', 'TPopulation1Jan': 'population'})
future_pop_data['population'] = future_pop_data['population'] * 1e3 # Convert from thousands to actual population
print('Population data cleaned.')

Population data cleaned.


## 4. Sector shares (Pauliuk 2008 + Harvey 2013 interpolation)

In [122]:
categories = ['vehicles', 'machinery', 'BC', 'appliances']

shares_1900 = {'vehicles': 0.179350518036637, 'machinery': 0.137017233159745,
               'BC': 0.604797209347797, 'appliances': 0.078835039455821}

shares_2008 = {cat: float(steel_shares[steel_shares['year'] == 2008][cat].values[0]) for cat in categories}

# Harvey (2022) Table 6 — redistributed to four categories
shares_2013_raw = {'vehicles': 0.079+0.020, 'machinery': 0.160+0.120,
                   'BC': 0.293+0.243, 'appliances': 0.031+0.030+0.007}
total = sum(shares_2013_raw.values())
shares_2013 = {k: v/total for k, v in shares_2013_raw.items()}

# Interpolate 2009-2013, hold constant at Harvey values from 2014 onward
future_share_years = list(range(2009, 2101))
future_shares_df   = pd.DataFrame(index=future_share_years, columns=categories)
for year in future_share_years:
    if year <= 2013:
        alpha = (year - 2008) / (2013 - 2008)
        for cat in categories:
            future_shares_df.loc[year, cat] = (1-alpha)*shares_2008[cat] + alpha*shares_2013[cat]
    else:
        for cat in categories:
            future_shares_df.loc[year, cat] = shares_2013[cat]
            
future_shares_df = future_shares_df.astype(float)
future_shares_df.index.name = 'year'
print('Sector shares prepared.')

Sector shares prepared.


## 5. Steel stock per capita — logistic projection (Watari L=11.3 t/cap)

In [112]:
steel_stock_cat = steel_stock_data[['year', 'steel_stock']].copy()
steel_stock_cat['steel_stock'] = steel_stock_cat['steel_stock'] * 1e3  # kt to t

stock_pre1900 = steel_stock_cat[steel_stock_cat['year'] < 1900][['year', 'steel_stock']].copy()
for cat, share in shares_1900.items():
    stock_pre1900[f'stock_{cat}'] = stock_pre1900['steel_stock'] * share

stock_post1900 = steel_stock_cat[(steel_stock_cat['year'] >= 1900) & (steel_stock_cat['year'] <= 2008)][['year', 'steel_stock']].copy()
stock_post1900 = stock_post1900.merge(steel_shares.reset_index(), on='year', how='left')
for cat in categories:
    stock_post1900[f'stock_{cat}'] = stock_post1900['steel_stock'] * stock_post1900[cat]

steel_stock_shares_hist = pd.concat([stock_pre1900, stock_post1900], ignore_index=True)
steel_stock_shares_hist = steel_stock_shares_hist.sort_values('year').reset_index(drop=True)
steel_stock_shares_hist = steel_stock_shares_hist[['year','steel_stock','stock_vehicles','stock_machinery','stock_BC','stock_appliances']]

steel_stock_pop = steel_stock_shares_hist.merge(pop_data, on='year', how='left')
steel_stock_pop['stock_per_capita'] = steel_stock_pop['steel_stock'] / steel_stock_pop['population']
steel_stock_pop = steel_stock_pop.dropna(subset=['stock_per_capita'])

In [117]:
def logistic_fixed_L(x, k, x0):
    L = 11.3 # Watari et al
    return L / (1 + np.exp(-k * (x - x0)))

recent = steel_stock_pop[steel_stock_pop['year'] >= 1800]

popt, _ = curve_fit(logistic_fixed_L,
                    np.array(recent['year']),
                    np.array(recent['stock_per_capita']),
                    p0=[0.05, 2030],
                    bounds=([0.01, 2020], [0.15, 2060]),
                    maxfev=10000)

k_fit, x0_fit = popt

print(f'Logistic fit: k={k_fit:.4f}, x0={x0_fit:.1f}')

last_year         = int(steel_stock_pop['year'].max())
future_years      = np.arange(last_year, 2101)
future_spc        = logistic_fixed_L(future_years, k_fit, x0_fit)
mask              = future_years > last_year
future_years_trim = future_years[mask]
future_spc_trim   = future_spc[mask]
future_pop_trim   = future_pop_data[future_pop_data['year'] > last_year]['population'].values
future_stock      = future_spc_trim * future_pop_trim

stock_df = pd.DataFrame({
    'year':             np.concatenate([steel_stock_pop['year'].values, future_years_trim]),
    'stock_per_capita': np.concatenate([steel_stock_pop['stock_per_capita'].values, future_spc_trim]),
    'total_stock':      np.concatenate([steel_stock_pop['steel_stock'].values, future_stock]),
    'population':       np.concatenate([steel_stock_pop['population'].values, future_pop_trim]),
    'type':             ['historic']*len(steel_stock_pop) +
                        ['recent' if y <= 2019 else 'predicted' for y in future_years_trim],
})
print(f'Stock 2008: {stock_df[stock_df["year"]==2008]["total_stock"].values[0]/1e9:.2f} Gt  (Pauliuk: 25 Gt)')
print(f'Stock 2050: {stock_df[stock_df["year"]==2050]["total_stock"].values[0]/1e9:.2f} Gt  (Pauliuk: ~60 Gt)')

print(stock_df)

Logistic fit: k=0.0211, x0=2040.0
Stock 2008: 25.72 Gt  (Pauliuk: 25 Gt)
Stock 2050: 60.23 Gt  (Pauliuk: ~60 Gt)
     year stock_per_capita         total_stock    population       type
0    1700         0.000008          4599.81792  5.954569e+08   historic
1    1710         0.000156         96661.12302  6.179759e+08   historic
2    1720         0.000423        273965.55415  6.481848e+08   historic
3    1730         0.000671        450249.70865  6.709000e+08   historic
4    1740         0.000892        626533.86315  7.022278e+08   historic
..    ...              ...                 ...           ...        ...
306  2096         8.644799  88443316652.916885  1.023081e+10  predicted
307  2097         8.687352  88792462118.674606  1.022089e+10  predicted
308  2098         8.729426  89129261482.768768  1.021021e+10  predicted
309  2099         8.771019  89453608610.377258  1.019877e+10  predicted
310  2100         8.812131  89765722153.547852  1.018661e+10  predicted

[311 rows x 5 columns]

## 6. Sector stock decomposition (with Harvey 2013 future shares)

In [118]:
steel_stock_cat2 = stock_df[['year', 'total_stock']].copy()

stock_pre1900b = steel_stock_cat2[steel_stock_cat2['year'] < 1900][['year', 'total_stock']].copy()
for cat, share in shares_1900.items():
    stock_pre1900b[f'stock_{cat}'] = stock_pre1900b['total_stock'] * share

stock_post1900b = steel_stock_cat2[(steel_stock_cat2['year'] >= 1900) & (steel_stock_cat2['year'] <= 2008)][['year', 'total_stock']].copy()
stock_post1900b = stock_post1900b.merge(steel_shares.reset_index(), on='year', how='left')
for cat in categories:
    stock_post1900b[f'stock_{cat}'] = stock_post1900b['total_stock'] * stock_post1900b[cat]

future_stock_cat = stock_df[stock_df['year'] > 2008][['year', 'total_stock']].copy()
for cat in categories:
    future_stock_cat[f'stock_{cat}'] = future_stock_cat.apply(
        lambda row: row['total_stock'] * (
            future_shares_df.loc[int(row['year']), cat]
            if int(row['year']) in future_shares_df.index
            else future_shares_df.loc[2013, cat]), axis=1)

steel_stock_shares = pd.concat([stock_pre1900b, stock_post1900b, future_stock_cat], ignore_index=True)
steel_stock_shares = steel_stock_shares.sort_values('year').reset_index(drop=True)
steel_stock_shares = steel_stock_shares[['year','total_stock','stock_vehicles','stock_machinery','stock_BC','stock_appliances']]
steel_stock_shares = steel_stock_shares[(steel_stock_shares['year'] >= 1800) & (steel_stock_shares['year'] <= 2100)].copy()
print('Sector stock decomposition done.')

Sector stock decomposition done.


## 7. DSM — logistic smoothing + stock-driven model per sector

In [125]:
t  = np.arange(1800, 2101)
CV = 0.3  # coefficient of variation (Pauliuk et al. 2013)

lt_vehicles   = {'Type': 'Normal', 'Mean': np.full(len(t), 20), 'StdDev': np.full(len(t), 20*CV)}
lt_machinery  = {'Type': 'Normal', 'Mean': np.full(len(t), 30), 'StdDev': np.full(len(t), 30*CV)}
lt_BC         = {'Type': 'Normal', 'Mean': np.full(len(t), 75), 'StdDev': np.full(len(t), 75*CV)}
lt_appliances = {'Type': 'Normal', 'Mean': np.full(len(t), 15), 'StdDev': np.full(len(t), 15*CV)}

def logistic(t, K, r, t0):
    return K / (1 + np.exp(-r * (t - t0)))

def fit_logistic_and_apply(t, stock_data):
    K_guess  = np.max(stock_data)
    r_guess  = 0.1
    t0_guess = t[np.argmax(stock_data >= K_guess / 2)]
    popt, _  = curve_fit(logistic, np.array(steel_stock_shares['year']),
                          stock_data, p0=(K_guess, r_guess, t0_guess))
    K_fit, r_fit, t0_fit = popt
    print(f'  K={K_fit:.2e}, r={r_fit:.4f}, t0={t0_fit:.1f}')
    return logistic(t, K_fit, r_fit, t0_fit)

def run_dsm(t, s, lt):
    dsm = DynamicStockModel(t=t, s=s, lt=lt)
    s_c, o_c, i = dsm.compute_stock_driven_model(NegativeInflowCorrect=False)
    o_c          = dsm.compute_o_c_from_s_c()
    o            = dsm.compute_outflow_total()
    stock_change = dsm.compute_stock_change()
    return s_c, o_c, o, stock_change, i

print('Fitting logistic and running DSM for each sector:')
s_vehicles   = fit_logistic_and_apply(t, np.array(steel_stock_shares['stock_vehicles']))
s_machinery  = fit_logistic_and_apply(t, np.array(steel_stock_shares['stock_machinery']))
s_BC         = fit_logistic_and_apply(t, np.array(steel_stock_shares['stock_BC']))
s_appliances = fit_logistic_and_apply(t, np.array(steel_stock_shares['stock_appliances']))

_, _, o_vehicles,   sc_vehicles,   i_vehicles   = run_dsm(t, s_vehicles,   lt_vehicles)
_, _, o_machinery,  sc_machinery,  i_machinery  = run_dsm(t, s_machinery,  lt_machinery)
_, _, o_BC,         sc_BC,         i_BC         = run_dsm(t, s_BC,         lt_BC    )
_, _, o_appliances, sc_appliances, i_appliances = run_dsm(t, s_appliances, lt_appliances)
print('DSM complete.')  

Fitting logistic and running DSM for each sector:
  K=1.09e+10, r=0.0284, t0=2042.4
  K=2.56e+10, r=0.0491, t0=2033.9
  K=5.62e+10, r=0.0318, t0=2039.3
  K=6.72e+09, r=0.0384, t0=2037.2
DSM complete.


## 8. Aggregate + yield losses

In [126]:
results_vehicles   = pd.DataFrame({'Year': t, 'stock_vehicles':   s_vehicles,   'inflow_vehicles':   i_vehicles,   'outflow_vehicles':   o_vehicles,   'stock_change_vehicles':   sc_vehicles})
results_machinery  = pd.DataFrame({'Year': t, 'stock_machinery':  s_machinery,  'inflow_machinery':  i_machinery,  'outflow_machinery':  o_machinery,  'stock_change_machinery':  sc_machinery})
results_BC         = pd.DataFrame({'Year': t, 'stock_BC':         s_BC,         'inflow_BC':         i_BC,         'outflow_BC':         o_BC,         'stock_change_BC':         sc_BC})
results_appliances = pd.DataFrame({'Year': t, 'stock_appliances': s_appliances, 'inflow_appliances': i_appliances, 'outflow_appliances': o_appliances, 'stock_change_appliances': sc_appliances})

results = (results_vehicles.merge(results_machinery, on='Year')
                            .merge(results_BC,        on='Year')
                            .merge(results_appliances, on='Year'))

results['total_inflow']       = results['inflow_vehicles']  + results['inflow_machinery']  + results['inflow_BC']  + results['inflow_appliances']
results['total_outflow']      = results['outflow_vehicles'] + results['outflow_machinery'] + results['outflow_BC'] + results['outflow_appliances']
results['total_stock_change'] = results['stock_change_vehicles'] + results['stock_change_machinery'] + results['stock_change_BC'] + results['stock_change_appliances']

for cat in categories:
    results[f'steel_needed_{cat}'] = results[f'inflow_{cat}'] / (0.92 * 0.96)
results['total_steel_needed'] = sum(results[f'steel_needed_{cat}'] for cat in categories)
results = results[results['Year'] != 1800].copy()
print('Results aggregated.')

Results aggregated.


## 9. Scrap supply + primary steel demand

In [127]:
def scrap_collection_s_curve(years, start=0.70, end=0.90, midpoint=2035, k=0.15):
    s = 1 / (1 + np.exp(-k * (years - midpoint)))
    s_min, s_max = s[0], s[-1]
    return start + (end - start) * (s - s_min) / (s_max - s_min)

def s_curve(years, start_pct, end_pct, midpoint, steepness=0.3):
    s = 1 / (1 + np.exp(-steepness * (years - midpoint)))
    s_min, s_max = s[0], s[-1]
    return start_pct + (end_pct - start_pct) * (s - s_min) / (s_max - s_min)

results_total = (results_vehicles.merge(results_machinery, on='Year')
                                  .merge(results_BC,        on='Year')
                                  .merge(results_appliances, on='Year'))
results_total['total_inflow']  = results_total['inflow_vehicles']  + results_total['inflow_machinery']  + results_total['inflow_BC']  + results_total['inflow_appliances']
results_total['total_outflow'] = results_total['outflow_vehicles'] + results_total['outflow_machinery'] + results_total['outflow_BC'] + results_total['outflow_appliances']
results_total['total_stock']   = results_total['stock_vehicles']   + results_total['stock_machinery']   + results_total['stock_BC']   + results_total['stock_appliances']

future = results_total[results_total['Year'] >= 2022].copy()
years_future = np.array(future['Year'])
future['scrap_collection_rate'] = scrap_collection_s_curve(years_future)
future['scrap_supply']          = future['total_outflow']
future['primary_steel_needed']  = np.maximum(future['total_inflow'] - future['scrap_supply'], 0)
print(f"Primary steel 2050: {future[future['Year']==2050]['primary_steel_needed'].values[0]/1e6:.0f} Mt")

Primary steel 2050: 846 Mt


## 10. DRI penetration scenarios

In [130]:
years = np.arange(2022, 2101)

scenarios_primary = {
    'Worst Case':  (0.02, 0.10, 2042, 0.25),
    'Circular Economy':  (0.02, 0.40, 2040, 0.28),
    'Technology-Led':  (0.02, 0.70, 2038, 0.30),
    'Best Case': (0.02, 1.00, 2034, 0.35),
}

primary_steel      = future['primary_steel_needed'].values
production_results = {}
for scenario, (s0, s1, mid, k) in scenarios_primary.items():
    dri_share = s_curve(years, s0, s1, mid, k)
    dri_prod  = primary_steel * dri_share
    bf_prod   = primary_steel * (1 - dri_share)
    production_results[scenario] = {
        'DRI Production': dri_prod, 'BF Production': bf_prod,
        'EAF Scrap': future['scrap_supply'].values,
        'Total': dri_prod + bf_prod + future['scrap_supply'].values
    }
print('Production scenarios ready.')

Production scenarios ready.


## 11. Emissions factors + results

In [131]:
MW_Fe = 55.845; MW_C = 12.011; MW_O = 15.999; MW_CO2 = MW_C + 2*MW_O

# BF-BOF stoichiometric + excess factor (WorldSteel / IEA Iron & Steel Roadmap)
CO2_reduction_BF        = (3 * MW_CO2) / (4 * MW_Fe)
excess_factor_BF        = 3.1
emissions_factor_BF     = CO2_reduction_BF * excess_factor_BF  # ~1.85 t CO2/t
emissions_factor_DRI_fossil = (3 * MW_CO2) / (2 * MW_Fe)
emissions_factor_DRI_green  = 0.0

# EAF scrap
elec_per_steel_EAF   = 0.600  # MWh/t
grid_CO2_intensity   = 0.500  # t CO2/MWh
electrode_rate       = 0.002  # t graphite/t
emissions_factor_EAF = elec_per_steel_EAF * grid_CO2_intensity + electrode_rate * (MW_CO2/MW_C)
print(f'BF-BOF: {emissions_factor_BF:.2f} t CO2/t steel')
print(f'EAF:    {emissions_factor_EAF:.2f} t CO2/t steel')

emissions_results = {}
for scenario in scenarios_primary:
    bf   = production_results[scenario]['BF Production']  / 1e6
    dri  = production_results[scenario]['DRI Production'] / 1e6
    eaf  = production_results[scenario]['EAF Scrap']      / 1e6
    emissions_results[scenario] = {
        'BF Emissions':        bf  * emissions_factor_BF,
        'DRI Emissions':       dri * emissions_factor_DRI_fossil + dri * emissions_factor_EAF,
        'EAF Emissions':       eaf * emissions_factor_EAF,
        'Green DRI Emissions': dri * emissions_factor_DRI_green  + dri * emissions_factor_EAF,
        'Total Emissions':     bf  * emissions_factor_BF + dri * emissions_factor_DRI_fossil + eaf * emissions_factor_EAF,
    }
print('Emissions calculated.')

BF-BOF: 1.83 t CO2/t steel
EAF:    0.31 t CO2/t steel
Emissions calculated.


## 12. Hydrogen demand + electrolyzer market shares

In [132]:
DRI_conversion = 54 / 1000  # t H2/t steel (stoichiometric, Perrone et al. 2025)

h2_results = {}
for scenario in scenarios_primary:
    h2_results[scenario] = {'H2 Demand (Mt)': production_results[scenario]['DRI Production'] * DRI_conversion / 1e6}

electrolyzer_shares = pd.DataFrame({
    'Year': [2022, 2030, 2050],
    'AEC Share':   [0.55, 0.35, 0.30],
    'PEM Share':   [0.35, 0.57, 0.60],
    'Other Share': [0.10, 0.08, 0.10]
})

plot_years   = future['Year'].values
interp_aec   = interp1d(electrolyzer_shares['Year'], electrolyzer_shares['AEC Share'],   kind='linear', fill_value='extrapolate')
interp_pem   = interp1d(electrolyzer_shares['Year'], electrolyzer_shares['PEM Share'],   kind='linear', fill_value='extrapolate')
interp_other = interp1d(electrolyzer_shares['Year'], electrolyzer_shares['Other Share'], kind='linear', fill_value='extrapolate')
aec_share  = interp_aec(plot_years)
pem_share  = interp_pem(plot_years)
other_share = interp_other(plot_years)

AEC_share_2022 = 0.55; PEM_share_2022 = 0.35; other_share_2022 = 0.10
for scenario in scenarios_primary:
    h2 = h2_results[scenario]['H2 Demand (Mt)']
    h2_results[scenario]['AEC H2']   = h2 * AEC_share_2022
    h2_results[scenario]['PEM H2']   = h2 * PEM_share_2022
    h2_results[scenario]['Other H2'] = h2 * other_share_2022
print('H2 demand ready.')

H2 demand ready.


## 13. Electrolyzer capacity + DSM + critical materials

In [133]:
available_hours     = (8760 - 11*24) * (1 - 0.03)
available_hours_AEC = available_hours_PEM = available_hours_other = available_hours

stack_lifetime_AEC   = 80000 / available_hours  # ~9.7 yr
stack_lifetime_PEM   = 65500 / available_hours  # ~8.0 yr
stack_lifetime_Other = 34500 / available_hours

efficiency_AEC = 56.7; efficiency_PEM = 60.5; efficiency_Other = 43.4

lt_AEC   = {'Type': 'Normal', 'Mean': np.full(len(years), round(stack_lifetime_AEC,   1)), 'StdDev': np.full(len(years), round(stack_lifetime_AEC,   1)*CV)}
lt_PEM   = {'Type': 'Normal', 'Mean': np.full(len(years), round(stack_lifetime_PEM,   1)), 'StdDev': np.full(len(years), round(stack_lifetime_PEM,   1)*CV)}
lt_Other = {'Type': 'Normal', 'Mean': np.full(len(years), 15.0),                           'StdDev': np.full(len(years), 15.0*CV)}

nickel_intensity = 800.0; platinum_intensity = 0.5; iridium_intensity = 0.75

cap_results = {}; dsm_results = {}; materials = {}
for scenario in scenarios_primary:
    cap_aec   = h2_results[scenario]['AEC H2']   * 1e9 * efficiency_AEC   / available_hours_AEC   / 1e6
    cap_pem   = h2_results[scenario]['PEM H2']   * 1e9 * efficiency_PEM   / available_hours_PEM   / 1e6
    cap_other = h2_results[scenario]['Other H2'] * 1e9 * efficiency_Other / available_hours_other / 1e6
    cap_results[scenario] = {'AEC': cap_aec, 'PEM': cap_pem, 'Other': cap_other}

    _, _, o_aec,   sc_aec,   i_aec   = run_dsm(years, cap_aec,   lt_AEC)
    _, _, o_pem,   sc_pem,   i_pem   = run_dsm(years, cap_pem,   lt_PEM)
    _, _, o_other, sc_other, i_other = run_dsm(years, cap_other, lt_Other)
    dsm_results[scenario] = {'cap_AEC': cap_aec, 'i_AEC': i_aec, 'o_AEC': o_aec,
                              'cap_PEM': cap_pem, 'i_PEM': i_pem, 'o_PEM': o_pem,
                              'cap_Other': cap_other, 'i_Other': i_other, 'o_Other': o_other}
    materials[scenario] = {
        'nickel_inflow':   i_aec * 1e3 * nickel_intensity   / 1e6,
        'nickel_stock':    cap_aec * 1e3 * nickel_intensity   / 1e6,
        'nickel_outflow':  o_aec * 1e3 * nickel_intensity   / 1e6,
        'platinum_inflow': i_pem * 1e3 * platinum_intensity / 1e6,
        'platinum_stock':  cap_pem * 1e3 * platinum_intensity / 1e6,
        'iridium_inflow':  i_pem * 1e3 * iridium_intensity  / 1e6,
        'iridium_stock':   cap_pem * 1e3 * iridium_intensity  / 1e6,
    }
print('Electrolyzer capacity + materials ready.')

Electrolyzer capacity + materials ready.


## 14. Save all results

In [134]:
stock_df.to_csv(output_folder / 'stock_df.csv', index=False)
results.to_csv(output_folder / 'results.csv', index=False)
future.to_csv(output_folder / 'future.csv', index=False)
steel_stock_pop.to_csv(output_folder / 'steel_stock_pop.csv', index=False)

with open(output_folder / 'model_outputs.pkl', 'wb') as f:
    pickle.dump({
        'scenarios_primary': scenarios_primary, 'years': years, 'plot_years': plot_years,
        'production_results': production_results, 'emissions_results': emissions_results,
        'emissions_factor_BF': emissions_factor_BF, 'emissions_factor_EAF': emissions_factor_EAF,
        'electrode_rate': electrode_rate, 'MW_CO2': MW_CO2, 'MW_C': MW_C,
        'h2_results': h2_results, 'electrolyzer_shares': electrolyzer_shares,
        'aec_share': aec_share, 'pem_share': pem_share, 'other_share': other_share,
        'cap_results': cap_results, 'dsm_results': dsm_results, 'materials': materials,
        'nickel_intensity': nickel_intensity, 'platinum_intensity': platinum_intensity, 'iridium_intensity': iridium_intensity,
        'efficiency_AEC': efficiency_AEC, 'efficiency_PEM': efficiency_PEM,
        'available_hours_AEC': available_hours_AEC, 'available_hours_PEM': available_hours_PEM,
        's_vehicles': s_vehicles, 's_machinery': s_machinery, 's_bc': s_bc, 's_appliances': s_appliances,
        'lt_vehicles': lt_vehicles, 'lt_machinery': lt_machinery, 'lt_bc': lt_bc, 'lt_appliances': lt_appliances,
        'k_fit': k_fit, 'x0_fit': x0_fit,
        'wsa_production': {1950:189,1955:270,1960:347,1965:456,1970:595,1975:644,1980:717,
                           1985:719,1990:770,1995:753,2000:850,2005:1148,2010:1435,2011:1540,
                           2012:1563,2013:1654,2014:1676,2015:1626,2016:1634,2017:1738,
                           2018:1831,2019:1879,2020:1883,2021:1963,2022:1889,2023:1904,2024:1885},
    }, f)

print('Saved to:', output_folder)
print('  CSVs: stock_df, results, future, steel_stock_pop')
print('  PKL:  model_outputs.pkl')

Saved to: c:\Users\ovid\MasterThesis\master\electrolysers\data\processed
  CSVs: stock_df, results, future, steel_stock_pop
  PKL:  model_outputs.pkl
